In [7]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
from sklearn.pipeline import Pipeline

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocessing import create_preprocessing_pipeline

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

Pandas version: 3.0.5
Numpy version: 2.5.1


## 1. Load Raw Train/Test Splits

To initiate the modeling workflow, we import the raw feature matrices and target vectors from the central `data/splits/` directory. The preprocessing pipeline will transform these raw features during model fitting and prediction.

* **Training partition:** `X_train_raw.csv` supplies the feature matrix, while `y_train_raw.csv` supplies the target vector.
* **Holdout partition:** `X_test_raw.csv` and `y_test_raw.csv` are loaded separately for final evaluation.
* **Check performed:** The following cell prints the dimensions of all four loaded objects.

In [8]:
# Load Raw Split dataset
X_train = pd.read_csv("../data/splits/X_train_raw.csv")
X_test = pd.read_csv("../data/splits/X_test_raw.csv")
y_train = pd.read_csv("../data/splits/y_train_raw.csv").squeeze("columns")
y_test = pd.read_csv("../data/splits/y_test_raw.csv").squeeze("columns")
# Inspect all datasets
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)



X_train shape: (16000, 33)
X_test shape: (4000, 33)
y_train shape: (16000,)
y_test shape: (4000,)


### 🔍 Dataset Loading & Schema Verification

The loading cell reports the following partition sizes:

* **Training features (`X_train`):** $16,000$ rows by $33$ columns.
* **Test features (`X_test`):** $4,000$ rows by $33$ columns.
* **Training targets (`y_train`):** $16,000$ labels.
* **Test targets (`y_test`):** $4,000$ labels.

> **Status:** The pre-split feature and target data are loaded and ready for cross-validation and model fitting.

## Cross-Validation Evaluation



Before training our final baseline model on the complete training set, we perform **5-fold cross-validation** to assess model generalization and stability across different subsets of $X_{\text{train}}$.

* **5-Fold Split ($k=5$):** $X_{\text{train}}$ ($16,000$ samples) is partitioned into $5$ equal folds ($3,200$ samples per fold). The model trains on $4$ folds ($12,800$ samples) and validates on the remaining fold in $5$ sequential iterations.
* **Metric Choice (`scoring="accuracy"`):** Measures the proportion of correctly classified loan statuses in each validation fold.
* **Data Leakage Isolation:** Cross-validation is executed strictly on $X_{\text{train}}$ and $y_{\text{train}}$. The holdout test set ($X_{\text{test}}$, $y_{\text{test}}$) remains entirely untouched.

In [9]:
# ============================================================
#  CROSS-VALIDATION
# ============================================================


# ------------------------------------------------------------
#  Instantiate the preprocessing pipeline and classifier
# ------------------------------------------------------------
# The preprocessor contains all data transformations such as
# imputation and categorical encoding.
#
# The classifier is the machine learning estimator that will
# learn to predict Loan_Status.
# ------------------------------------------------------------

preprocessor = create_preprocessing_pipeline()

classifier = DecisionTreeClassifier(
    random_state=42
)


# ------------------------------------------------------------
#  Combine preprocessing and classifier into one pipeline
# ------------------------------------------------------------
# Keeping preprocessing and the model inside the same pipeline
# ensures that preprocessing is fitted only on the training
# portion of each cross-validation fold.
#
# This prevents data leakage from the validation folds.
# ------------------------------------------------------------

model_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", classifier),
    ]
)


# ------------------------------------------------------------
#  Run 5-Fold Cross-Validation
# ------------------------------------------------------------
# The dataset is divided into 5 folds.
#
# For each fold:
#   1. Four folds are used for training.
#   2. One fold is used for validation.
#   3. The process is repeated until every fold has been
#      used as the validation set.
#
# Because the preprocessing and model are inside the same
# pipeline, fitting happens independently within each fold.
# This provides leakage-safe evaluation.
# ------------------------------------------------------------

cv_scores = cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
)


# ------------------------------------------------------------
#  Display the accuracy from each fold
# ------------------------------------------------------------

print("=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)

print("\nAccuracy for each fold:")

for fold_number, score in enumerate(cv_scores, start=1):
    print(f"Fold {fold_number}: {score:.4f}")


# ------------------------------------------------------------
#  Calculate the mean cross-validation accuracy
# ------------------------------------------------------------
# The mean represents the average accuracy achieved across
# all 5 validation folds.
# ------------------------------------------------------------

mean_cv_score = cv_scores.mean()

print("\nMean Cross-Validation Accuracy:")
print(f"{mean_cv_score:.4f}")


# ------------------------------------------------------------
#  Calculate the standard deviation
# ------------------------------------------------------------
# Standard deviation measures how much the accuracy varies
# between the different cross-validation folds.
#
# A smaller value generally indicates more consistent model
# performance across the folds.
# ------------------------------------------------------------

std_cv_score = cv_scores.std()

print("\nStandard Deviation of CV Accuracy:")
print(f"{std_cv_score:.4f}")


# ------------------------------------------------------------
#  Display the final cross-validation summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CROSS-VALIDATION SUMMARY")
print("=" * 60)

print(f"Number of folds: {len(cv_scores)}")
print(f"Individual fold accuracies: {cv_scores}")
print(f"Mean CV accuracy: {mean_cv_score:.4f}")
print(f"Standard deviation: {std_cv_score:.4f}")

print("\nCross-validation completed successfully.")

CROSS-VALIDATION RESULTS

Accuracy for each fold:
Fold 1: 0.9413
Fold 2: 0.9375
Fold 3: 0.9369
Fold 4: 0.9375
Fold 5: 0.9328

Mean Cross-Validation Accuracy:
0.9372

Standard Deviation of CV Accuracy:
0.0027

CROSS-VALIDATION SUMMARY
Number of folds: 5
Individual fold accuracies: [0.94125   0.9375    0.936875  0.9375    0.9328125]
Mean CV accuracy: 0.9372
Standard deviation: 0.0027

Cross-validation completed successfully.


### 📊 Cross-Validation Results & Fold Breakdown

The 5-fold cross-validation results show consistent performance across all validation subsets:

#### Fold Accuracy Breakdown
* **Fold 1:** $0.9413$ ($94.13\%$)
* **Fold 2:** $0.9375$ ($93.75\%$)
* **Fold 3:** $0.9369$ ($93.69\%$)
* **Fold 4:** $0.9375$ ($93.75\%$)
* **Fold 5:** $0.9328$ ($93.28\%$)

#### Overall Performance Summary

| Metric | Value | Interpretation |
| :--- | :--- | :--- |
| **Mean CV Accuracy** | **$0.9372$ ($93.72\%$)** | High average baseline performance across all validation folds |
| **Standard Deviation ($\sigma$)** | **$0.0027$ ($0.27\%$)** | Low variation across the five folds |

> **Key Takeaway:** The scores range from $93.28\%$ to $94.13\%$, with a standard deviation of $0.0027$. This indicates consistent validation performance before fitting the model on the full $X_{\text{train}}$ dataset.

**Notes:**

Cross-validation helps us determine whether the Decision Tree is performing reliably or just happened to perform well on one particular split of the training data.

Cross-validation is used to check how consistently the model performs across different portions of the training data.

Instead of relying on one train/validation split, it tests the model multiple times and gives us an average performance score.

##  Final Model Training


Having validated baseline model stability through 5-fold cross-validation, we fit the `DecisionTreeClassifier` on the complete training set ($X_{\text{train}}$, $y_{\text{train}}$).

* **Maximizing Learning:** Fitting across $100\%$ of the training data ($16,000$ samples) allows the tree to build split rules using the maximum amount of information available.
* **Preserving Integrity:** The test set ($X_{\text{test}}$, $y_{\text{test}}$) remains strictly segregated for out-of-sample evaluation.

In [10]:


# Train the Decision Tree model using the entire training dataset.
# X_train contains the input features.
# y_train contains the target variable (Loan_Status).

model_pipeline.fit(X_train, y_train)


# ------------------------------------------------------------
# Display training completion message
# ------------------------------------------------------------

print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

print("\nDecision Tree model trained successfully.")

print(f"Number of training samples: {X_train.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")

MODEL TRAINING

Decision Tree model trained successfully.
Number of training samples: 16000
Number of features: 33


### 🎯 Model Training Summary

The `DecisionTreeClassifier` was successfully fitted on the full training partition:

* **Training Samples:** $16,000$ rows ($80.00\%$ of total dataset)
* **Raw Feature Space:** $33$ predictor variables. The pipeline expands categorical features internally before they reach the classifier.
* **Model Status:** Fitted and ready for predictions on unseen holdout data (`X_test`)

> **Next Step:** Generate predictions on $X_{\text{test}}$ to evaluate out-of-sample generalization metrics including accuracy, precision, recall, F1-score, and the confusion matrix.

## Final Model Evaluation



To quantify out-of-sample generalization, we generate predictions using the holdout test set ($X_{\text{test}}$, $y_{\text{test}}$), which was strictly isolated from training and cross-validation.

* **Generalization Check:** Compares performance on unseen data against the $5$-fold cross-validation mean ($93.72\%$) to check for a meaningful performance gap.
* **Evaluation Framework:** Uses **Accuracy**, **Confusion Matrix**, and **Classification Report** (Precision, Recall, F1-Score) to assess class-level predictive quality.

### Metrics to use for Evaluation

The `classification_report()` function from Scikit-Learn generates a concise text summary of the key evaluation metrics for each class in a classification model. It provides a quick, comprehensive view of how well your model distinguishes between categories, making it far more informative than overall accuracy alone—especially for imbalanced datasets.

---

### Key Metrics Explained

* **Precision:** The ratio of correct positive predictions to the total predicted positives ($\frac{TP}{TP + FP}$). It answers: *Of all instances the model predicted as Class X, how many were actually Class X?* High precision means low false-positive rate.
* **Recall (Sensitivity):** The ratio of correct positive predictions to all actual positive instances ($\frac{TP}{TP + FN}$). It answers: *Of all actual Class X instances in the dataset, how many did the model capture?* High recall means low false-negative rate.
* **F1-Score:** The harmonic mean of Precision and Recall ($2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$). It provides a single balance score between precision and recall, peaking at $1.0$ (perfect performance) and dropping toward $0$.
* **Support:** The actual count of ground-truth occurrences for each class in the dataset. It helps spot class imbalance and ensures metric scores are interpreted with context.

---

### Summary Aggregates

Instead of calling individual evaluation functions like `precision_score()`, `recall_score()`, and `f1_score()` separately, `classification_report()` conveniently presents all core evaluation metrics together in a single consolidated output:

* **Precision:** Calculates the accuracy of positive predictions using `precision_score()`.
* **Recall:** Measures the ability to capture all positive instances using `recall_score()`.
* **F1-Score:** Computes the harmonic mean of precision and recall using `f1_score()`.
* **Support:** Shows the structural count of true instances per class.

It also displays the model's overall accuracy, alongside macro and weighted averages, giving you a complete overview without running multiple function calls.

In [11]:
# ============================================================
# FINAL EVALUATION
# ============================================================



# ------------------------------------------------------------
# Make predictions on the unseen test data
# ------------------------------------------------------------
# The model has already been trained using X_train and y_train.
# Now we give it X_test, which it has never seen before.
# ------------------------------------------------------------

y_pred = model_pipeline.predict(X_test)


# ------------------------------------------------------------
# Calculate the test accuracy
# ------------------------------------------------------------
# Accuracy tells us the percentage of test predictions
# that the model got correct.
# ------------------------------------------------------------

test_accuracy = accuracy_score(y_test, y_pred)


# ------------------------------------------------------------
# Display the final accuracy
# ------------------------------------------------------------

print("=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy (%): {test_accuracy * 100:.2f}%")


# ------------------------------------------------------------
# Display the confusion matrix
# ------------------------------------------------------------
# The confusion matrix shows:
#
# True Negatives  → Correctly predicted negative class
# False Positives → Negative class predicted as positive
# False Negatives → Positive class predicted as negative
# True Positives  → Correctly predicted positive class
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(y_test, y_pred)

print(cm)


# ------------------------------------------------------------
# Display the classification report
# ------------------------------------------------------------
# The classification report provides:
#
# Precision → How many predicted positives were actually positive
# Recall    → How many actual positives were correctly identified
# F1-score  → Balance between precision and recall
# Support   → Number of actual samples in each class
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(classification_report(y_test, y_pred))

FINAL MODEL EVALUATION

Test Accuracy: 0.9345
Test Accuracy (%): 93.45%

CONFUSION MATRIX
[[2428  143]
 [ 119 1310]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Approved       0.95      0.94      0.95      2571
    Rejected       0.90      0.92      0.91      1429

    accuracy                           0.93      4000
   macro avg       0.93      0.93      0.93      4000
weighted avg       0.93      0.93      0.93      4000



### 📊 Test Evaluation Results & Interpretation

The Decision Tree model achieved **$93.45\%$ accuracy** on the unseen holdout test data. This is $0.27$ percentage points below the 5-fold cross-validation mean ($93.72\%$), indicating similar validation and test performance.

---

#### 📌 Confusion Matrix Breakdown ($N = 4,000$)

| | Predicted: Approved | Predicted: Rejected | Total Actual |
| :--- | :---: | :---: | :---: |
| **Actual: Approved** | **2,428** | **143** | **2,571** |
| **Actual: Rejected** | **119** | **1,310** | **1,429** |

* **Correct approvals:** $2,428$ approved applications were predicted as approved.
* **Correct rejections:** $1,310$ rejected applications were predicted as rejected.
* **Incorrect approvals:** $119$ rejected applications were predicted as approved.
* **Incorrect rejections:** $143$ approved applications were predicted as rejected.

---

#### 🎯 Classification Performance Metrics

* **Overall Accuracy ($93.45\%$):** $3,738$ out of $4,000$ test cases were correctly classified.
* **Rejected Loans:**
  * **Precision ($0.90$):** When the model predicts a rejection, it is correct $90\%$ of the time.
  * **Recall ($0.92$):** The model successfully identifies $92\%$ of all actual rejections.
  * **F1-Score ($0.91$):** Strong balance between precision and recall for the minority class.
* **Approved Loans:**
  * **Precision ($0.95$):** When the model predicts an approval, it is correct $95\%$ of the time.
  * **Recall ($0.94$):** The model captures $94\%$ of all actual loan approvals.
  * **F1-Score ($0.95$):** Highly reliable performance on the majority class.

---

#### 💡 Key Takeaways

1. **Validation-to-test consistency:** Test accuracy ($93.45\%$) is close to the cross-validation mean ($93.72\%$), differing by $0.27$ percentage points.
2. **Error distribution:** The model produced $119$ incorrect approvals and $143$ incorrect rejections; these counts should be evaluated against the business cost of each error type.
3. **Class-level performance:** The classification report gives rejected loans precision, recall, and F1-score of $0.90$, $0.92$, and $0.91$; approved loans round to $0.95$, $0.94$, and $0.95$.

## Save Trained Model



To persist our trained `DecisionTreeClassifier` for downstream deployment, API integration, or batch inference, we serialize the model object to disk using `joblib`.

* **Path Traversal (`../models/`):** Navigates up from `notebooks/` into the central `models/` folder to separate model artifacts from code and data assets.
* **Serialization Choice (`joblib`):** Efficiently serializes large Python data structures and scikit-learn models containing heavy NumPy arrays.

In [ ]:
# ============================================================
# SAVE TRAINED MODEL
# ============================================================




# Save the trained Decision Tree model
joblib.dump(model_pipeline, "../models/loan_approval_pipeline.pkl")


# Confirm that the model was saved successfully
print("=" * 60)
print("MODEL SAVING")
print("=" * 60)

print("\nModel saved successfully.")
print("File: loan_approval_pipeline.pkl")

MODEL SAVING

Model saved successfully.
File: loan_approval_decision_tree.pkl


### 💾 Model Serialization Summary

The trained Decision Tree model artifact has been written to disk:

* **Saved Artifact:** `models/loan_approval_decision_tree.pkl`
* **Model Configuration:** `DecisionTreeClassifier(random_state=42)`
* **Deployment Readiness:** The `.pkl` binary can now be loaded directly into inference pipelines using `joblib.load()` without retraining.

> **Project Milestone Reached:** The end-to-end Machine Learning pipeline—spanning data cleaning, feature engineering, train-test splitting, cross-validation, model training, test evaluation, and artifact serialization—is complete!